In [2]:
import sys
from pathlib import Path

# Add parent directory ("../") to sys.path
sys.path.append(str(Path("../").resolve()))

In [3]:
from app.models.embedding import Embedding
from app.models.repo_file import RepoFile
from sqlalchemy import select
from app.rag.embedder.embedder import Embedder
from app.core.db import AsyncSessionLocal
from app.services.ingest.ingest_service import Ingest
import uuid
from fastembed.rerank.cross_encoder import TextCrossEncoder
from sqlalchemy import func
from openai import OpenAI
from app.core.config import env_config

/home/user/Documents/Project_5/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Preset 'granite_vision_v4' already registered for ChartExtractionVlmEngineOptions


In [3]:
reranker = TextCrossEncoder(model_name="jinaai/jina-reranker-v1-turbo-en")

In [6]:
query = "How are tasks created and then how are they saved to the DB"

In [4]:
groq_connector = OpenAI(api_key=env_config.GROQ_API_KEY, base_url="https://api.groq.com/openai/v1",)

In [12]:
new_query = groq_connector.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[
        {
            "role": "system",
            "content": "You are a profesional prompt writer, who is specialized in writing prompt for the RAG system and you need to rewrite the prompt given to you. The prompt is supposed to do retrival from a database, your role is to rewrite the prompt in a way that the retrival becomes way better then what it is now, the user will give a prompt you will enhance it and even use some other word terminolgies so that the correct retrival can be done, just return the prompt and nothing else should be returned by you. Rembember just a good prompt, nothing else. Maintain the user's original intent but vastly improve clarity and depth.",
        },
        {"role": "user", "content": query},
    ],
)

[INFO] httpx2: HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [13]:
new_prompt = new_query.choices[0].message.content
print(new_prompt)

Retrieve information regarding the task creation workflow and persistence mechanisms. Specifically, identify the steps involved in defining new tasks, including input validation and default attribute assignment, and detail the process for committing these objects to the database. Include information on the specific database operations (e.g., INSERT, ORM save methods), transaction management, data integrity constraints, and any triggers or middleware that hook into the task saving lifecycle.


In [34]:
embedder = Embedder()

[INFO] app.rag.embedder.embedder: [EMBEDDER] SUCCESSFULLY LOADED THE EMBEDDING MODEL: BAAI/bge-small-en-v1.5


In [35]:
user_query_vector = embedder.embed(query)
ai_query_vector = embedder.embed(new_prompt)

In [36]:
searcher_1 = list(user_query_vector)[0].tolist()
seracher_2 = list(ai_query_vector)[0].tolist()

In [37]:
distance = Embedding.vector.cosine_distance(searcher_1).label("distance")
vector_1_stmt = (
    select(
        Embedding.embed_id,
        Embedding.file_id,
        RepoFile.file_path,
        Embedding.chunk,
        Embedding.chunk_str,
        distance,
    )
    .join(RepoFile, RepoFile.file_id == Embedding.file_id)
    .order_by(distance)
    .limit(20)
)

In [38]:
distance = Embedding.vector.cosine_distance(seracher_2).label("distance")
vector_2_stmt = (
    select(
        Embedding.embed_id,
        Embedding.file_id,
        RepoFile.file_path,
        Embedding.chunk,
        Embedding.chunk_str,
        distance,
    )
    .join(RepoFile, RepoFile.file_id == Embedding.file_id)
    .order_by(distance)
    .limit(20)
)

In [39]:
lexical_score = func.similarity(Embedding.chunk_str, query).label("similarity")
lexical_stmt = (
    select(
        Embedding.embed_id,
        Embedding.file_id,
        RepoFile.file_path,
        Embedding.chunk,
        Embedding.chunk_str,
        lexical_score,
    )
    .join(RepoFile, RepoFile.file_id == Embedding.file_id)
    .where(func.similarity(Embedding.chunk_str, query) > 0.05)  # Filter out completely irrelevant text
    .order_by(lexical_score.desc())
    .limit(20)
)

In [14]:
lexical_score_2 = func.similarity(Embedding.chunk_str, new_prompt).label("similarity")
lexical_stmt_2 = (
    select(
        Embedding.embed_id,
        Embedding.file_id,
        RepoFile.file_path,
        Embedding.chunk,
        Embedding.chunk_str,
        lexical_score_2,
    )
    .join(RepoFile, RepoFile.file_id == Embedding.file_id)
    .where(func.similarity(Embedding.chunk_str, query) > 0.05)  # Filter out completely irrelevant text
    .order_by(lexical_score_2.desc())
    .limit(20)
)

In [40]:
ts_query = func.plainto_tsquery("english", query)
fts_score = func.ts_rank(
    func.to_tsvector("english", Embedding.chunk_str),
    ts_query
).label("fts_score")

# 3. Build the full-text search statement
fts_stmt = (
    select(
        Embedding.embed_id,
        Embedding.file_id,
        RepoFile.file_path,
        Embedding.chunk,
        Embedding.chunk_str,
        fts_score,
    )
    .join(RepoFile, RepoFile.file_id == Embedding.file_id)
    # Perform FTS match: to_tsvector(chunk_str) @@ plainto_tsquery(query)
    .where(func.to_tsvector("english", Embedding.chunk_str).op("@@")(ts_query))
    .order_by(fts_score.desc())
    .limit(20)
)

In [16]:
ts_query_2 = func.plainto_tsquery("english", new_prompt)
fts_score_2 = func.ts_rank(
    func.to_tsvector("english", Embedding.chunk_str),
    ts_query_2
).label("fts_score_")

# 3. Build the full-text search statement
fts_stmt_2 = (
    select(
        Embedding.embed_id,
        Embedding.file_id,
        RepoFile.file_path,
        Embedding.chunk,
        Embedding.chunk_str,
        fts_score_2,
    )
    .join(RepoFile, RepoFile.file_id == Embedding.file_id)
    # Perform FTS match: to_tsvector(chunk_str) @@ plainto_tsquery(query)
    .where(func.to_tsvector("english", Embedding.chunk_str).op("@@")(ts_query_2))
    .order_by(fts_score_2.desc())
    .limit(20)
)

In [41]:
async with AsyncSessionLocal() as db:
    vector_res_1 = await db.execute(vector_1_stmt)
    vector_res_2 = await db.execute(vector_2_stmt)
    lexical_res = await db.execute(lexical_stmt)
    fts_res = await db.execute(fts_stmt)
    lexical_res_2 = await db.execute(lexical_stmt_2)
    fts_res_2 = await db.execute(fts_stmt_2)

    vector_records_1 = vector_res_1.mappings().all()
    vector_records_2 = vector_res_2.mappings().all()
    lexical_records = lexical_res.mappings().all()
    fts_records = fts_res.mappings().all()
    lexical_records_2 = lexical_res_2.mappings().all()
    fts_records_2 = fts_res_2.mappings().all()

In [42]:
from collections import defaultdict

def merge_results_rrf_multi(result_lists: list, k: int = 60, id_key: str = "embed_id") -> list[dict]:
    rrf_scores = {}
    doc_map = {}

    for record_list in result_lists:
        for rank, record in enumerate(record_list, start=1):
            doc_id = record[id_key]
            if doc_id not in doc_map:
                doc_map[doc_id] = record
            
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k + rank))

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    final_records = []
    for doc_id, score in sorted_docs:
        # Convert SQLAlchemy RowMapping / Row to a plain python dict
        rec = dict(doc_map[doc_id])
        rec["rrf_score"] = score
        final_records.append(rec)

    return final_records

In [43]:
# vector_records_1 = [dict(row) for row in vector_res_1.mappings().all()]
# vector_records_2 = [dict(row) for row in vector_res_2.mappings().all()]
# lexical_records = [dict(row) for row in lexical_res.mappings().all()]

final_records = merge_results_rrf_multi(
    [vector_records_1, vector_records_2, lexical_records, fts_records,], 
    k=60, 
    id_key="embed_id"
)

In [ ]:
# Format passages for Jina cross-encoder
passages = [
    # f"File: {r['file_path']}\n"
    f"Symbol: {r['chunk'].get('name', '')}\n"
    f"Code:\n{r['chunk_str']}"
    for r in final_records
]

# Rerank the unified candidate pool
scores = list(reranker.rerank(query, passages))

# Pair, sort, and slice to top 20
# final_results = sorted(
#     zip(candidate_records, scores),
#     key=lambda pair: pair[1],
#     reverse=True
# )[:20]

In [21]:
import math
def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))

# 4. Attach both raw and normalized scores back to final records
for rec, raw_score in zip(final_records, scores):
    rec["rerank_score_raw"] = float(raw_score)
    rec["rerank_score"] = round(sigmoid(float(raw_score)), 4)

final_records.sort(key=lambda x: x["rerank_score"], reverse=True)

In [45]:
# --- Print Results ---
content = []
for rank, record in enumerate(final_records, start=1):
    chunk = record.get("chunk")
    rerank_score = record.get("rerank_score", 0.0)

    # Safely retrieve original search metrics
    orig_dist = record.get("distance")
    orig_sim = record.get("similarity")

    # Format original score display depending on which query retrieved the chunk
    if orig_dist is not None:
        score_info = f"vector_distance={orig_dist:.4f}"
    elif orig_sim is not None:
        score_info = f"trigram_similarity={orig_sim:.4f}"
    else:
        score_info = "score_n/a"

    # Safely retrieve keys with fallbacks to avoid KeyErrors
    embed_id = record.get("embed_id") or record.get("embedding_id", "N/A")
    file_id = record.get("file_id", "N/A")
    repo_file_path = record.get("file_path") or record.get("repo_file_path", "N/A")
    chunk_str = record.get("chunk_str", "")

    print(f"\n#{rank} rerank_score={rerank_score:.4f} ({score_info})")
    print(f"embedding_id={embed_id} file_id={file_id}")
    print(f"repo_file_path={repo_file_path}")
    content.append(
        f"#{rank} rerank_score={rerank_score:.4f} ({score_info})\nembedding_id={embed_id} file_id={file_id}\nrepo_file_path={repo_file_path}\nchunk_str={chunk_str}\n"
    )
    print(f"chunk_str={chunk_str}")
    print(f"chunk={chunk}")


#1 rerank_score=0.0000 (vector_distance=0.2957)
embedding_id=ddab3806-93e0-42ac-9dd4-ccf7cca36f8d file_id=a337d661-1957-4e9e-a8c1-9c5fb9d1f789
repo_file_path=backend/src/services/task.services.ts
chunk_str=File: /home/user/Documents/Project_5/backend/_repo/backend/src/services/task.services.ts | Language: typescript | Kind: function | Name: createTask (part 2) | Code:                 list_id: data.list_id,
                position: numberOfTask + 1,
                created_at: Date.now(),
                updated_at: Date.now(),
            })
            // console.log(task)
            return {
                status: 200,
                message: "Task created successfully",
                data: task
            }
        } catch (error) {
            console.log(error)
            return { status: 500, message: "Internal Server Error" }
        }
    } catch (error) {
        console.log(error)
        return { status: 500, message: "Internal Server Error" }
    }
}
chunk={'kind':

In [23]:
gemeni_connector = OpenAI(api_key=env_config.GEMINI_API_KEY, base_url="https://generativelanguage.googleapis.com/v1beta/openai/",)

In [24]:
result = groq_connector.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "You are a professional assisant that work for the Repo-Decoder and your work is to help user with his query, for each of the query the documet chunks will be provided to you as the context and then you will look on the docs if the answer is found return that answer if not then return I'm not able to answer this at this moment. Answer to the user in the best way possible",
        },
        {
            "role": "user",
            "content": f'''
            context: {content[:15]}
            user_query: {query}
            '''
        }
    ],
)

[INFO] httpx2: HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


In [25]:
from IPython.display import display, Markdown
display(Markdown(result.choices[0].message.content))



**Task creation flow**

1. **Controller entry point** – `backend/src/controllers/task.controller.ts`  
   ```ts
   const createTask = async (req, res) => {
       const user_id = req.user?.id
       const { title, description, deadline, board_id, status, list_id } = req.body

       const result = await createTaskService({
           user_id,
           title,
           description,
           deadline,
           board_id,
           status,
           list_id
       })
       …
   }
   ```
   The controller extracts the data sent by the client (the request body and the authenticated user id) and forwards it to the **service layer**.

2. **Service logic** – `backend/src/services/task.services.ts` (`createTask`)  

   ```ts
   const createTask = async (data) => {
       // 1️⃣  Validate required fields
       if (!data.user_id || !data.title || !data.deadline ||
           !data.board_id || !data.status || !data.list_id) {
           return { status: 400, message: "All fields are required" }
       }

       // 2️⃣  Verify the board exists and belongs to the user
       const board = await Board.findOne({ user_id: data.user_id, _id: data.board_id })
       if (!board) return { status: 404, message: "Board not found" }

       // 3️⃣  Verify the target list exists inside that board
       const list = await List.findOne({ _id: data.list_id, board_id: data.board_id })
       if (!list) return { status: 404, message: "List not found" }

       // 4️⃣  Determine the new task’s position in the list
       const numberOfTask = await Task.countDocuments({
           board_id: data.board_id,
           list_id: data.list_id
       })

       // 5️⃣  **Create & persist the task**
       const task = await Task.create({
           user_id: data.user_id,
           title: data.title,
           description: data.description || '',
           deadline: data.deadline,
           board_id: data.board_id,
           status: data.status,
           list_id: data.list_id,
           position: numberOfTask + 1,
           created_at: Date.now(),
           updated_at: Date.now(),
       })
       …
   }
   ```

   * The function first checks that every required field is present.  
   * It then fetches the board and list to make sure they exist and belong to the user.  
   * `Task.countDocuments` is used to compute the next `position` value for the new card.  
   * Finally, **`Task.create`** (a Mongoose model operation) writes the new document to MongoDB.  
   * On success the service returns `{ status: 200, message: "Task created successfully", data: task }`.

3. **Mongoose model** – `backend/src/models/task.model.ts`

   ```ts
   const taskSchema = new mongoose.Schema({
       user_id:   { type: mongoose.Schema.Types.ObjectId, ref: "User", required: true },
       board_id:  { type: mongoose.Schema.Types.ObjectId, ref: "Board", required: true },
       title:     { type: String, required: true },
       description:{ type: String },
       deadline:  { type: Date, required: true },
       list_id:   { type: mongoose.Schema.Types.ObjectId, ref: "List", required: true },
       status:    { type: String, enum: ['Todo','In Progress','Completed'],
                    default: 'Todo', required: true },
       created_at:{ type: Date, default: Date.now, required: true },
       updated_at:{ type: Date, default: Date.now, required: true },
       position:  { type: Number, required: true }
   })
   const Task = mongoose.model('Task', taskSchema)
   ```

   The schema defines the shape of a task document, including `position`, timestamps, and the required references (`user_id`, `board_id`, `list_id`). When `Task.create` is called, Mongoose validates the data against this schema and inserts the document into the **tasks** collection in MongoDB.

---

### Summary
* The request hits the **controller**, which passes the payload to the **service**.  
* The **service** validates input, checks that the board and list exist, calculates the new task’s position, and then calls **`Task.create`**.  
* `Task.create` uses the **Mongoose `Task` model** (defined by `taskSchema`) to write a new task document to the MongoDB database.  
* Upon successful creation the controller returns a `200` response with the newly saved task data.

In [26]:
from app.models.file_imports import FileImport

stmt = select(FileImport)

async with AsyncSessionLocal() as db:
    result = await db.execute(stmt)
    mapp = result.mappings().all()

In [27]:
for data in mapp:
    obj = data.get("FileImport")
    # Exclude internal SQLAlchemy state key
    data_dict = {k: v for k, v in obj.__dict__.items() if not k.startswith('_')}
    print(data_dict)

{'file_id': UUID('e33ee776-4a67-48e2-bb6d-e6e850665bf0'), 'source': 'vite', 'symbols': [{'name': 'defineConfig', 'alias': None}], 'module': 'vite', 'is_static': False, 'import_id': UUID('d90ece2a-efca-483c-aa52-7eddd06023fd'), 'imported_file_id': None, 'module_alias': None, 'file_type': <FileType.EXTERNAL: 'external'>, 'resolved_path': '', 'is_wildcard': False}
{'file_id': UUID('e33ee776-4a67-48e2-bb6d-e6e850665bf0'), 'source': '@vitejs/plugin-react-swc', 'symbols': [{'name': 'react', 'alias': None}], 'module': '@vitejs/plugin-react-swc', 'is_static': False, 'import_id': UUID('640a1c1c-9fe5-442f-b29a-2b5af1a355f2'), 'imported_file_id': None, 'module_alias': None, 'file_type': <FileType.INTERNAL: 'internal'>, 'resolved_path': '', 'is_wildcard': False}
{'file_id': UUID('e33ee776-4a67-48e2-bb6d-e6e850665bf0'), 'source': '@tailwindcss/vite', 'symbols': [{'name': 'tailwindcss', 'alias': None}], 'module': '@tailwindcss/vite', 'is_static': False, 'import_id': UUID('f0a8bdaf-bb31-4d08-b76c-4ee